In [1]:
# --------------------------------------------------
# STEP 6.1 — MOBILENETV2 MODEL SETUP
# --------------------------------------------------

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# --------------------------------------------------
# Configuration
# --------------------------------------------------

NUM_CLASSES = 13

# --------------------------------------------------
# Select device
# --------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("=" * 60)
print("DEVICE INFORMATION")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA version:",
        torch.version.cuda
    )
else:
    print("CUDA GPU not available. Using CPU.")

# --------------------------------------------------
# Load ImageNet-pretrained MobileNetV2
# --------------------------------------------------

weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(weights=weights)

# --------------------------------------------------
# Replace final classifier
# --------------------------------------------------

input_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    input_features,
    NUM_CLASSES
)

# Move model to device
model = model.to(device)

# --------------------------------------------------
# Display model information
# --------------------------------------------------

print("\n" + "=" * 60)
print("MODEL INFORMATION")
print("=" * 60)

print("Architecture: MobileNetV2")
print("Pretrained weights: ImageNet")
print("Original classifier input features:", input_features)
print("Output classes:", NUM_CLASSES)

print("\nFinal classifier:")
print(model.classifier)

# --------------------------------------------------
# Test forward pass
# --------------------------------------------------

dummy_input = torch.randn(
    2,
    3,
    224,
    224
).to(device)

with torch.no_grad():
    output = model(dummy_input)

print("\n" + "=" * 60)
print("FORWARD PASS TEST")
print("=" * 60)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert output.shape == (2, NUM_CLASSES)

print("\n" + "=" * 60)
print("STEP 6.1 STATUS: PASSED")
print("=" * 60)

DEVICE INFORMATION
PyTorch version: 2.14.0+cpu
Device: cpu
CUDA GPU not available. Using CPU.
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\ATHARVA/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:01<00:00, 14.1MB/s]



MODEL INFORMATION
Architecture: MobileNetV2
Pretrained weights: ImageNet
Original classifier input features: 1280
Output classes: 13

Final classifier:
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=13, bias=True)
)

FORWARD PASS TEST
Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 13])

STEP 6.1 STATUS: PASSED


In [2]:
# --------------------------------------------------
# STEP 6.2 — LOAD DATA PIPELINE FOR TRAINING
# --------------------------------------------------

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# --------------------------------------------------
# Configuration
# --------------------------------------------------

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# --------------------------------------------------
# Training augmentation
# --------------------------------------------------

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1)
    ),

    transforms.RandomHorizontalFlip(
        p=0.5
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# --------------------------------------------------
# Validation / Test preprocessing
# --------------------------------------------------

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])

# --------------------------------------------------
# Dataset paths
# --------------------------------------------------

TRAIN_DIR = "data/processed/tomato_potato_split/train"
VAL_DIR = "data/processed/tomato_potato_split/val"
TEST_DIR = "data/processed/tomato_potato_split/test"

# --------------------------------------------------
# Create datasets
# --------------------------------------------------

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=val_test_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=val_test_transform
)

# --------------------------------------------------
# Create DataLoaders
# --------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

# --------------------------------------------------
# Display information
# --------------------------------------------------

print("=" * 60)
print("TRAINING DATA PIPELINE")
print("=" * 60)

print("\nTrain samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

print("\nClasses:", len(train_dataset.classes))

print("\nClass mapping:")
for i, class_name in enumerate(train_dataset.classes):
    print(f"{i}: {class_name}")

# --------------------------------------------------
# Verify class mapping consistency
# --------------------------------------------------

assert train_dataset.classes == val_dataset.classes
assert train_dataset.classes == test_dataset.classes

# --------------------------------------------------
# Test one batch
# --------------------------------------------------

images, labels = next(iter(train_loader))

print("\n" + "=" * 60)
print("BATCH TEST")
print("=" * 60)

print("Images:", images.shape)
print("Labels:", labels.shape)
print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert len(train_dataset) == 14208
assert len(val_dataset) == 3045
assert len(test_dataset) == 3045

assert images.shape == (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
assert labels.shape == (BATCH_SIZE,)

print("\n" + "=" * 60)
print("STEP 6.2 STATUS: PASSED")
print("=" * 60)

TRAINING DATA PIPELINE

Train samples: 14208
Validation samples: 3045
Test samples: 3045

Classes: 13

Class mapping:
0: Potato___Early_blight
1: Potato___Late_blight
2: Potato___healthy
3: Tomato___Bacterial_spot
4: Tomato___Early_blight
5: Tomato___Late_blight
6: Tomato___Leaf_Mold
7: Tomato___Septoria_leaf_spot
8: Tomato___Spider_mites Two-spotted_spider_mite
9: Tomato___Target_Spot
10: Tomato___Tomato_Yellow_Leaf_Curl_Virus
11: Tomato___Tomato_mosaic_virus
12: Tomato___healthy

BATCH TEST
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Image dtype: torch.float32
Label dtype: torch.int64

STEP 6.2 STATUS: PASSED


In [3]:
# --------------------------------------------------
# STEP 6.3 — TRANSFER LEARNING CONFIGURATION
# --------------------------------------------------

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --------------------------------------------------
# Configuration
# --------------------------------------------------

NUM_CLASSES = 13
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# --------------------------------------------------
# Device
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# --------------------------------------------------
# Load pretrained MobileNetV2
# --------------------------------------------------

weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(weights=weights)

# --------------------------------------------------
# Freeze feature extractor
# --------------------------------------------------

for param in model.features.parameters():
    param.requires_grad = False

# --------------------------------------------------
# Replace classifier
# --------------------------------------------------

input_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    input_features,
    NUM_CLASSES
)

model = model.to(device)

# --------------------------------------------------
# Calculate class weights
# --------------------------------------------------

train_labels = np.array([
    label for _, label in train_dataset.samples
])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print("\nClass weights:")

for index, (class_name, weight) in enumerate(
    zip(train_dataset.classes, class_weights)
):
    print(f"{index}: {class_name}: {weight.item():.4f}")

# --------------------------------------------------
# Loss function
# --------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# --------------------------------------------------
# Optimizer
# --------------------------------------------------

optimizer = torch.optim.Adam(
    model.classifier[1].parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# --------------------------------------------------
# Learning-rate scheduler
# --------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# --------------------------------------------------
# Count trainable parameters
# --------------------------------------------------

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("\n" + "=" * 60)
print("MODEL TRAINING CONFIGURATION")
print("=" * 60)

print("Architecture: MobileNetV2")
print("Pretrained: ImageNet")
print("Classes:", NUM_CLASSES)

print("\nLearning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)
print("Loss: Weighted CrossEntropyLoss")
print("Optimizer: Adam")
print("Scheduler: ReduceLROnPlateau")

print("\nTrainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert len(train_dataset.classes) == NUM_CLASSES
assert model.classifier[1].out_features == NUM_CLASSES

assert criterion.weight.shape == (NUM_CLASSES,)

assert all(
    param.requires_grad is False
    for param in model.features.parameters()
)

assert all(
    param.requires_grad
    for param in model.classifier[1].parameters()
)

print("\n" + "=" * 60)
print("STEP 6.3 STATUS: PASSED")
print("=" * 60)

Device: cpu

Class weights:
0: Potato___Early_blight: 1.5613
1: Potato___Late_blight: 1.5613
2: Potato___healthy: 10.3106
3: Tomato___Bacterial_spot: 0.7340
4: Tomato___Early_blight: 1.5613
5: Tomato___Late_blight: 0.8211
6: Tomato___Leaf_Mold: 1.6410
7: Tomato___Septoria_leaf_spot: 0.8814
8: Tomato___Spider_mites Two-spotted_spider_mite: 0.9317
9: Tomato___Target_Spot: 1.1118
10: Tomato___Tomato_Yellow_Leaf_Curl_Virus: 0.2914
11: Tomato___Tomato_mosaic_virus: 4.1874
12: Tomato___healthy: 0.9855

MODEL TRAINING CONFIGURATION
Architecture: MobileNetV2
Pretrained: ImageNet
Classes: 13

Learning rate: 0.001
Weight decay: 0.0001
Loss: Weighted CrossEntropyLoss
Optimizer: Adam
Scheduler: ReduceLROnPlateau

Trainable parameters: 16653
Total parameters: 2240525

STEP 6.3 STATUS: PASSED


In [4]:
# --------------------------------------------------
# STEP 6.4 — TRAINING LOOP SMOKE TEST
# --------------------------------------------------

import time
import torch

print("=" * 60)
print("TRAINING LOOP SMOKE TEST")
print("=" * 60)

# --------------------------------------------------
# Training mode
# --------------------------------------------------

model.train()

smoke_losses = []

start_time = time.time()

# --------------------------------------------------
# Run only 3 batches
# --------------------------------------------------

for batch_idx, (images, labels) in enumerate(train_loader):

    if batch_idx >= 3:
        break

    batch_start = time.time()

    # Move data to device
    images = images.to(device)
    labels = labels.to(device)

    # Clear gradients
    optimizer.zero_grad()

    # Forward pass
    outputs = model(images)

    # Calculate loss
    loss = criterion(outputs, labels)

    # Backpropagation
    loss.backward()

    # Update classifier
    optimizer.step()

    smoke_losses.append(loss.item())

    batch_time = time.time() - batch_start

    print(
        f"Batch {batch_idx + 1}/3 | "
        f"Loss: {loss.item():.4f} | "
        f"Time: {batch_time:.2f}s"
    )

# --------------------------------------------------
# Validation inference test
# --------------------------------------------------

model.eval()

with torch.no_grad():

    val_images, val_labels = next(iter(val_loader))

    val_images = val_images.to(device)
    val_labels = val_labels.to(device)

    val_outputs = model(val_images)

    val_loss = criterion(
        val_outputs,
        val_labels
    )

    val_predictions = torch.argmax(
        val_outputs,
        dim=1
    )

# --------------------------------------------------
# Results
# --------------------------------------------------

total_time = time.time() - start_time

print("\n" + "=" * 60)
print("SMOKE TEST RESULTS")
print("=" * 60)

print("Training batches completed:", len(smoke_losses))
print("Training losses:", [
    round(loss, 4)
    for loss in smoke_losses
])

print("\nValidation batch:")
print("Images:", val_images.shape)
print("Labels:", val_labels.shape)
print("Outputs:", val_outputs.shape)
print("Validation loss:", round(val_loss.item(), 4))
print("Predictions:", val_predictions.shape)

print("\nTotal smoke-test time:", round(total_time, 2), "seconds")

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert len(smoke_losses) == 3
assert val_outputs.shape == (BATCH_SIZE, NUM_CLASSES)
assert val_predictions.shape == (BATCH_SIZE,)
assert torch.isfinite(val_loss)

print("\n" + "=" * 60)
print("STEP 6.4 STATUS: PASSED")
print("=" * 60)

TRAINING LOOP SMOKE TEST
Batch 1/3 | Loss: 2.5131 | Time: 0.63s
Batch 2/3 | Loss: 2.5374 | Time: 0.49s
Batch 3/3 | Loss: 2.5962 | Time: 0.50s

SMOKE TEST RESULTS
Training batches completed: 3
Training losses: [2.5131, 2.5374, 2.5962]

Validation batch:
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Outputs: torch.Size([32, 13])
Validation loss: 2.2491
Predictions: torch.Size([32])

Total smoke-test time: 4.39 seconds

STEP 6.4 STATUS: PASSED


In [5]:
# --------------------------------------------------
# STEP 6.5 — FINAL TRAINING CONFIGURATION
# --------------------------------------------------

import random
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

# --------------------------------------------------
# Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("REPRODUCIBILITY")
print("=" * 60)
print("Random seed:", SEED)

# --------------------------------------------------
# Device
# --------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

# --------------------------------------------------
# Training configuration
# --------------------------------------------------

NUM_CLASSES = 13
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3

# --------------------------------------------------
# Checkpoint directory
# --------------------------------------------------

CHECKPOINT_DIR = Path("models/mobilenetv2")

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR.resolve())

# --------------------------------------------------
# Fresh MobileNetV2
# --------------------------------------------------

weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(
    weights=weights
)

# --------------------------------------------------
# Freeze feature extractor
# --------------------------------------------------

for param in model.features.parameters():
    param.requires_grad = False

# --------------------------------------------------
# Replace classifier
# --------------------------------------------------

input_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    input_features,
    NUM_CLASSES
)

model = model.to(device)

# --------------------------------------------------
# Class weights
# --------------------------------------------------

train_labels = np.array([
    label
    for _, label in train_dataset.samples
])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_labels
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

# --------------------------------------------------
# Loss
# --------------------------------------------------

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# --------------------------------------------------
# Optimizer
# --------------------------------------------------

optimizer = torch.optim.Adam(
    model.classifier[1].parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# --------------------------------------------------
# Scheduler
# --------------------------------------------------

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

# --------------------------------------------------
# Training history
# --------------------------------------------------

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

# --------------------------------------------------
# Early stopping state
# --------------------------------------------------

best_val_loss = float("inf")
epochs_without_improvement = 0

# --------------------------------------------------
# Count parameters
# --------------------------------------------------

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

print("\n" + "=" * 60)
print("FINAL TRAINING CONFIGURATION")
print("=" * 60)

print("Architecture: MobileNetV2")
print("Pretrained weights: ImageNet")
print("Number of classes:", NUM_CLASSES)

print("\nBatch size:", BATCH_SIZE)
print("Maximum epochs:", MAX_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

print("\nLoss: Weighted CrossEntropyLoss")
print("Optimizer: Adam")
print("Scheduler: ReduceLROnPlateau")
print("Early stopping patience:", EARLY_STOPPING_PATIENCE)

print("\nTrainable parameters:", trainable_parameters)
print("Total parameters:", total_parameters)

# --------------------------------------------------
# Validation
# --------------------------------------------------

assert model.classifier[1].out_features == NUM_CLASSES

assert all(
    not p.requires_grad
    for p in model.features.parameters()
)

assert all(
    p.requires_grad
    for p in model.classifier[1].parameters()
)

assert criterion.weight.shape == (NUM_CLASSES,)

assert best_val_loss == float("inf")
assert epochs_without_improvement == 0

print("\n" + "=" * 60)
print("STEP 6.5 STATUS: PASSED")
print("=" * 60)

REPRODUCIBILITY
Random seed: 42

Device: cpu

Checkpoint directory:
C:\Users\ATHARVA\Desktop\Mini project\ai-farmer-assistant\models\mobilenetv2

FINAL TRAINING CONFIGURATION
Architecture: MobileNetV2
Pretrained weights: ImageNet
Number of classes: 13

Batch size: 32
Maximum epochs: 10
Learning rate: 0.001
Weight decay: 0.0001

Loss: Weighted CrossEntropyLoss
Optimizer: Adam
Scheduler: ReduceLROnPlateau
Early stopping patience: 3

Trainable parameters: 16653
Total parameters: 2240525

STEP 6.5 STATUS: PASSED


In [ ]:
# --------------------------------------------------
# STEP 6.6 — MOBILENETV2 BASELINE TRAINING
# --------------------------------------------------

import time
import copy
import torch

# --------------------------------------------------
# Fresh model check
# --------------------------------------------------

model.train()

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": [],
    "learning_rate": []
}

print("=" * 70)
print("MOBILENETV2 BASELINE TRAINING")
print("=" * 70)

print(f"Device: {device}")
print(f"Maximum epochs: {MAX_EPOCHS}")
print(f"Early stopping patience: {EARLY_STOPPING_PATIENCE}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

training_start = time.time()

# --------------------------------------------------
# Epoch loop
# --------------------------------------------------

for epoch in range(MAX_EPOCHS):

    epoch_start = time.time()

    # ==================================================
    # TRAINING
    # ==================================================

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_accuracy = correct / total

    # ==================================================
    # VALIDATION
    # ==================================================

    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            predictions = torch.argmax(outputs, dim=1)

            val_correct += (
                predictions == labels
            ).sum().item()

            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    # ==================================================
    # Scheduler
    # ==================================================

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]["lr"]

    # ==================================================
    # Store history
    # ==================================================

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)
    history["learning_rate"].append(current_lr)

    # ==================================================
    # Checkpoint / Early stopping
    # ==================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": best_model_state,
                "val_loss": best_val_loss,
                "val_accuracy": val_accuracy,
                "classes": train_dataset.classes
            },
            CHECKPOINT_DIR / "best_model.pth"
        )

        checkpoint_status = "BEST"

    else:

        epochs_without_improvement += 1

        checkpoint_status = ""

    epoch_time = time.time() - epoch_start

    # ==================================================
    # Print epoch results
    # ==================================================

    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy * 100:.2f}% | "
        f"LR: {current_lr:.6f} | "
        f"{epoch_time:.1f}s "
        f"{checkpoint_status}"
    )

    # ==================================================
    # Early stopping
    # ==================================================

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:

        print(
            f"\nEarly stopping triggered after "
            f"{epoch + 1} epochs."
        )

        break

# --------------------------------------------------
# Restore best model
# --------------------------------------------------

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

total_training_time = time.time() - training_start

# --------------------------------------------------
# Final training summary
# --------------------------------------------------

best_epoch = (
    history["val_loss"].index(best_val_loss) + 1
)

best_val_accuracy = history["val_accuracy"][best_epoch - 1]

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Epochs completed:", len(history["train_loss"]))

print("Best epoch:", best_epoch)

print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)

print(
    "Best validation accuracy:",
    f"{best_val_accuracy * 100:.2f}%"
)

print(
    "Total training time:",
    f"{total_training_time / 60:.2f} minutes"
)

print(
    "\nBest checkpoint:",
    CHECKPOINT_DIR / "best_model.pth"
)

print("\n" + "=" * 70)
print("STEP 6.6 STATUS: TRAINING COMPLETE")
print("=" * 70)